In [34]:
import os


In [35]:
%pwd

'c:\\PROJECTS\\End_to_end_Kidney_Disease_Classification_Deep_learning_project'

In [4]:
os.chdir("../")

In [5]:
%pwd

'c:\\PROJECTS\\End_to_end_Kidney_Disease_Classification_Deep_learning_project'

In [36]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir:Path
    source_url: str
    local_data_file:Path
    unzip_dir:Path


In [38]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml,create_directories


In [39]:
class ConfigurationManager:
    def __init__(self,config_filepath=CONFIG_FILE_PATH,params_filepath=PARAMS_FILE_PATH):
        print(config_filepath)
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self)-> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config=DataIngestionConfig(
            root_dir = config.root_dir,
            source_url = config.source_url,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )

        return data_ingestion_config
    


In [40]:
import os
import zipfile
import gdown
from cnnClassifier import logger


In [31]:
import os
import gdown
from src.cnnClassifier import logger

class DataIngestion:
    def __init__(self, config):
        self.config = config

    def download_file(self) -> str:
        try:
            dataset_url = self.config.source_url
            zip_download_dir = self.config.local_data_file

            os.makedirs(os.path.dirname(zip_download_dir), exist_ok=True)

            logger.info("Downloading data from Google Drive")

            # ✅ Case 1: if full URL is given
            if "http" in dataset_url:
                file_id = dataset_url.split("/d/")[1].split("/")[0]
            else:
                # ✅ Case 2: already a file ID
                file_id = dataset_url

            url = f"https://drive.google.com/uc?id={file_id}"

            gdown.download(url, str(zip_download_dir), quiet=False)

            logger.info(f"Downloaded file to {zip_download_dir}")

        except Exception as e:
            logger.error("Download failed")
            raise 

In [41]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self) -> str:
        try:
            dataset_url = self.config.source_url
            zip_download_dir = self.config.local_data_file

            os.makedirs("artifacts/data_ingestion", exist_ok=True)

            logger.info("Downloading data from Google Drive")

            file_id = dataset_url.split("/")[-2]

            prefix = "https://drive.google.com/uc?export=download&id="

            gdown.download(
                prefix + file_id,
                str(zip_download_dir),
                quiet=False
            )

            logger.info(
                f"Downloaded data from {dataset_url} into file {zip_download_dir}"
            )

        except Exception as e:
            raise e
        

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path,exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file,"r") as zip_ref:
            zip_ref.extractall(unzip_path)


In [42]:
    try:
        config = ConfigurationManager()
        data_ingestion_config = config.get_data_ingestion_config()
        data_ingestion=DataIngestion(config=data_ingestion_config)
        data_ingestion.download_file()
        data_ingestion.extract_zip_file()
    except Exception as e:
        raise e

    

config\config.yaml
[2026-05-08 10:55:27,641: INFO: common: Yaml file loaded successfully]
[2026-05-08 10:55:27,645: INFO: common: Yaml file loaded successfully]
[2026-05-08 10:55:27,650: INFO: common: Created Directory:artifacts]
[2026-05-08 10:55:27,653: INFO: common: Created Directory:artifacts/data_ingestion]
[2026-05-08 10:55:27,657: INFO: 3381671941: Downloading data from Google Drive]


Downloading...
From (original): https://drive.google.com/uc?id=190TPBZMV8J__LYKfNpdWAqMzdqRDY6HP
From (redirected): https://drive.google.com/uc?id=190TPBZMV8J__LYKfNpdWAqMzdqRDY6HP&confirm=t&uuid=aaef0d87-b4e7-4fe1-81da-78bfae6a1c61
To: c:\PROJECTS\End_to_end_Kidney_Disease_Classification_Deep_learning_project\artifacts\data_ingestion\data.zip
100%|██████████| 940M/940M [00:15<00:00, 61.2MB/s] 

[2026-05-08 10:55:45,140: INFO: 3381671941: Downloaded data from https://drive.google.com/file/d/190TPBZMV8J__LYKfNpdWAqMzdqRDY6HP/view?usp=sharing into file artifacts/data_ingestion/data.zip]
